# Lab 10 · Multimodal retrieval

**Day 3 · S16, lab 3 of 3** · Budget: 25 min of the 30 min slot · Runs on: Colab with a T4 GPU (a laptop works, slower)

Your lab 07 pipeline only sees text, but OQ's documents are full of diagrams and scanned forms. This lab puts the 20 images from labs 08 and 09 into a search index in two ways, then measures which one finds the right document.

| Pattern | How an image gets into the index | Models at ingest |
|---|---|---|
| **A. Shared embedding space** | embed the pixels with CLIP; embed the query with CLIP's text encoder | CLIP |
| **B. Caption at ingest** | a vision model writes a searchable description, which is embedded like any other text chunk | vision model + text embedder |

Then the answer step: find the document by its text, then send the original image to the vision model to answer. **Retrieve on the caption, answer from the pixels.**

## 1. Setup

On a fresh Colab runtime this installs Ollama, downloads the self-hosted vision model, and downloads two embedding models (CLIP, about 600 MB, and `bge-small`, about 130 MB). Start it now and read ahead.

This lab uses one vision model: the self-hosted one if it's available, otherwise the vendor API.

In [ ]:
# Setup: find the lab folder, detect the runtime, install pinned packages on Colab.
import os
import subprocess
import sys
from pathlib import Path

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

REPO_URL = os.environ.get("LAB_REPO_URL", "")  # Colab: the course repo URL, once it is published


def find_root():
    for p in [Path.cwd(), *Path.cwd().parents]:
        if (p / "scripts" / "vision_client.py").exists():
            return p


ROOT = find_root()
if ROOT is None and IN_COLAB and REPO_URL:
    subprocess.run(["git", "clone", "-q", REPO_URL, "/content/lab"], check=True)
    ROOT = Path("/content/lab")
if ROOT is None:
    raise RuntimeError("Lab folder not found. Open this notebook from inside it, or set LAB_REPO_URL on Colab.")
if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "openai==3.0.0", "python-dotenv==1.1.0"], check=True)
sys.path.insert(0, str(ROOT / "scripts"))
print("lab folder:", ROOT, "| runtime:", "Colab" if IN_COLAB else "local")

In [ ]:
from vision_client import ensure_ollama, load_openai_key, prebaked_models, run_batch, self_hosted, vendor_api

RUN_MODE = os.environ.get("LAB_RUN_MODE", "live")  # "live" calls the models, "prebaked" replays a saved run
LAB = "10"
OUT = ROOT / "outputs" / LAB
PREBAKED = Path(os.environ.get("LAB_PREBAKED_DIR", ROOT / "facilitator" / "prebaked_outputs")) / LAB

VLM = None
if RUN_MODE == "live":
    try:
        ensure_ollama()
        VLM = self_hosted()
    except Exception as e:
        print("self-hosted model unavailable:", e)
        if load_openai_key(ROOT):
            VLM = vendor_api()
if VLM is None:
    baked = prebaked_models(PREBAKED / "captions")
    VLM = baked[0] if baked else None
    print("using prebaked outputs from", PREBAKED)
assert VLM, f"No live model and no prebaked outputs in {PREBAKED}. Ask the facilitator."
print("vision model:", VLM.label)

## 2. The documents

There are 20 images: 10 diagrams from lab 08 and 10 work orders from lab 09. The four text chunks stand in for the lab 07 corpus; in the capstone, the image documents go into the lab 07 index instead.

In [ ]:
import hashlib
import json
from dataclasses import dataclass

import numpy as np
import pandas as pd
from PIL import Image
from render_images import build_image_set

MANIFEST = ROOT / "corpus" / "images" / "manifest.csv"
if not MANIFEST.exists():
    build_image_set(ROOT)
manifest = pd.read_csv(MANIFEST)

IMAGE_DOCS = [{"doc_id": r.item_id, "modality": "image", "image": ROOT / r.image, "text": None}
              for r in manifest.itertuples()]

TEXT_DOCS = [
    {"doc_id": "txt_01", "modality": "text", "image": None,
     "text": "Procedure MP-K301-07, Compressor K-301 isolation. Before opening the compressor casing: obtain a "
             "permit to work, lock out the motor breaker, close and lock the suction and discharge valves, "
             "depressurise to flare through the blowdown valve, and confirm a zero gas reading with a portable detector."},
    {"doc_id": "txt_02", "modality": "text", "image": None,
     "text": "Integration standard IS-04. Field applications reach the ERP only through the Integration Hub. "
             "Direct database connections to the ERP are not permitted. The API Gateway authenticates callers "
             "with the Identity Provider using OAuth2."},
    {"doc_id": "txt_03", "modality": "text", "image": None,
     "text": "Alarm handling guideline AG-02. Priority 1 alarms page the on-call engineer within two minutes. "
             "Every alarm raised by the Alarm Server is published to the Event Bus, and a ticket is opened "
             "in the ITSM Platform."},
    {"doc_id": "txt_04", "modality": "text", "image": None,
     "text": "Historian retention policy HR-01. Raw process tags are kept in the Plant Historian for 90 days, "
             "then exported nightly over SFTP to the Data Lake for long-term storage."},
]
print(len(IMAGE_DOCS), "images,", len(TEXT_DOCS), "text chunks")

## 3. The index

Both patterns use the same index. The only difference is the pair of embedding functions you give it: one for documents, one for queries. Each hit keeps the path to the original image, which the answer step needs.

In [ ]:
@dataclass
class Hit:
    doc_id: str
    score: float
    modality: str  # "image" or "text"
    text: str | None  # the text chunk, or the image's caption
    image: Path | None  # the original image, kept for the answer step


class MultimodalIndex:
    """add() documents, search() returns ranked hits. The two embed functions decide the pattern.

    TODO(contract 5): match the lab 07 index interface in docs/contracts.md.
    """

    def __init__(self, embed_docs, embed_query):
        self.embed_docs, self.embed_query = embed_docs, embed_query
        self.docs, self.vectors = [], None

    def add(self, docs):
        vectors = self.embed_docs(docs)
        self.vectors = vectors if self.vectors is None else np.vstack([self.vectors, vectors])
        self.docs += docs

    def search(self, query, k=5):
        scores = self.vectors @ self.embed_query(query)
        return [Hit(self.docs[i]["doc_id"], round(float(scores[i]), 3), self.docs[i]["modality"],
                    self.docs[i]["text"], self.docs[i]["image"]) for i in np.argsort(-scores)[:k]]


def table(hits):
    return pd.DataFrame([{"doc_id": h.doc_id, "modality": h.modality, "score": h.score} for h in hits])

## 4. Pattern A: CLIP

CLIP was trained on photos paired with captions, and it puts images and text in the same vector space. So you can embed the pixels directly, with no vision model at ingest.

In [ ]:
from sentence_transformers import SentenceTransformer

clip = SentenceTransformer("clip-ViT-B-32")


def clip_docs(docs):
    return clip.encode([Image.open(d["image"]).convert("RGB") for d in docs], normalize_embeddings=True)


def clip_query(query):
    return clip.encode(query, normalize_embeddings=True)


index_a = MultimodalIndex(clip_docs, clip_query)
index_a.add(IMAGE_DOCS)
table(index_a.search("work order for compressor K-301"))

Is wo_01, the K-301 work order, at the top? Look at how close together the scores are. CLIP can tell a form from a diagram, but it reads very little of what's written on them.

## 5. Pattern B: caption at ingest

A vision model writes a description of each image, including all the text it can read. The description is embedded with an ordinary text embedder (`bge-small`), the same kind lab 07 uses. The captions are cached in `outputs/10/captions/`.

In [ ]:
CAPTION_PROMPT = """Describe this image for a search index.
First say what kind of document it is, for example: IT architecture diagram, process flow diagram, maintenance work order form.
Then write out the text you can read: the title, every box or system name, equipment tags, labels on arrows,
and for a form every filled-in field and reading. Use plain sentences. Do not guess at text you cannot read."""

caption_jobs = [{"item_id": d["doc_id"], "image": d["image"]} for d in IMAGE_DOCS]
captions = run_batch(VLM, caption_jobs, prompt=CAPTION_PROMPT, out_dir=OUT / "captions",
                     prebaked_dir=PREBAKED / "captions", workers=4 if VLM.backend == "openai" else 1)
CAPTION_DOCS = [{**d, "text": c["output"] or ""} for d, c in zip(IMAGE_DOCS, captions)]

for d in CAPTION_DOCS[:1] + CAPTION_DOCS[10:11]:
    print(f"--- {d['doc_id']}\n{d['text'][:600]}\n")

In [ ]:
bge = SentenceTransformer("BAAI/bge-small-en-v1.5")
QUERY_PREFIX = "Represent this sentence for searching relevant passages: "  # bge's instruction for queries


def text_docs(docs):
    return bge.encode([d["text"] for d in docs], normalize_embeddings=True)


def text_query(query):
    return bge.encode(QUERY_PREFIX + query, normalize_embeddings=True)


index_b = MultimodalIndex(text_docs, text_query)
index_b.add(CAPTION_DOCS)
table(index_b.search("work order for compressor K-301"))

## 6. Which pattern finds the right document?

Eight questions with known answers. The tier 2 and tier 3 images are copies of the same content, so any copy counts as a hit. **hit@3** is the share of questions where a right document is in the top 3. **MRR** (mean reciprocal rank) is 1.0 when the right document is always first.

In [ ]:
QUERIES = [
    ("Which work order was raised for compressor K-301?", {"wo_01", "wo_04", "wo_07"}),
    ("What protocol does the OPC gateway use to send data to the plant historian?", {"dia_02"}),
    ("Where does recycle valve FV-310 send gas back to?", {"dia_05", "dia_08"}),
    ("Which systems receive events from the event bus?", {"dia_06", "dia_09"}),
    ("Vibration reading on booster pump P-101A during the quarterly PM", {"wo_02", "wo_05", "wo_08"}),
    ("How does the field tablet app reach the ERP?", {"dia_01", "dia_04", "dia_07"}),
    ("Heat exchanger E-401 outlet temperature too high", {"wo_03", "wo_06", "wo_09"}),
    ("How are booster pumps P-101A and P-101B arranged?", {"dia_10"}),
]


def evaluate(index, k=3):
    rows = []
    for query, relevant in QUERIES:
        ranked = [h.doc_id for h in index.search(query, k=len(index.docs))]
        first = next(rank for rank, doc_id in enumerate(ranked, 1) if doc_id in relevant)
        rows.append({"query": query, "rank": first, "hit": first <= k, "rr": 1 / first})
    return pd.DataFrame(rows)


res_a, res_b = evaluate(index_a), evaluate(index_b)
print(f"hit@3   CLIP {res_a.hit.mean():.2f}   caption {res_b.hit.mean():.2f}")
print(f"MRR     CLIP {res_a.rr.mean():.2f}   caption {res_b.rr.mean():.2f}")
pd.DataFrame({"query": res_a["query"], "CLIP rank": res_a["rank"], "caption rank": res_b["rank"]})

When pattern B misses, open the caption for that image (`CAPTION_DOCS`). A caption can't contain text the vision model couldn't read, and the tier 3 copies are where it struggles most.

## 7. One index for text and images

Pattern B turns every image into text, so the images go into the same index as your text chunks, through the same embedder and the same query path. Each hit still records whether it came from an image.

You can't do this with CLIP vectors. They sit in a different space from your text embedder, and even inside CLIP, text-to-text scores run higher than text-to-image scores, so text chunks would crowd the images out.

In [ ]:
index_mixed = MultimodalIndex(text_docs, text_query)
index_mixed.add(CAPTION_DOCS + TEXT_DOCS)

for query in ["How does the field tablet app reach the ERP?", "What happens when the alarm server raises an alarm?"]:
    print(query)
    display(table(index_mixed.search(query, k=4)))

## 8. Answer from the pixels

The caption got the document found. For the answer, go back to the source: if the top hit is an image, send the original image and the question to the vision model. A caption is a summary and may have left out the one detail the question needs.

In [ ]:
ANSWER_PROMPT = """Answer the question using only the {source} provided. Be brief.
If the answer is not there, reply: Not in this document.

Question: {question}"""


def answer(question, index=index_mixed):
    top = index.search(question, k=1)[0]
    if top.modality == "image":
        prompt, image = ANSWER_PROMPT.format(source="image", question=question), top.image
    else:
        prompt, image = ANSWER_PROMPT.format(source="document", question=question) + f"\n\nDocument:\n{top.text}", None
    job = {"item_id": "answer_" + hashlib.sha1(question.encode()).hexdigest()[:10], "image": image, "prompt": prompt}
    rec = run_batch(VLM, [job], out_dir=OUT / "answers", prebaked_dir=PREBAKED / "answers", verbose=False)[0]
    return top, rec["output"] or rec["error"]


for question, expected in [
    ("What protocol does the OPC gateway use to send data to the plant historian?", "MQTT"),
    ("Where does recycle valve FV-310 send gas back to?", "Inlet Separator V-201"),
    ("What was the vibration reading on booster pump P-101A during the quarterly PM?", "2.1 mm/s"),
    ("What must be done before opening the compressor casing?",
     "permit, lock out the breaker, lock the valves, depressurise, gas test"),
]:
    top, reply = answer(question)
    print(f"Q: {question}\n   source:   {top.doc_id} ({top.modality})\n   answer:   {reply}\n   expected: {expected}\n")

## 9. What to take away

- **For documents, captioning at ingest beats a shared embedding space.** The value of a diagram or a form is the text written on it, and CLIP reads very little of it.
- **Captions inherit the vision model's mistakes.** A bad scan gives a bad caption, and the document becomes impossible to find, with no warning. Score your captions the way you scored extraction in labs 08 and 09.
- **Keep the image path next to the caption.** Retrieve on the text, answer from the pixels, and cite the image so a person can check it.
- **CLIP-style embeddings earn their place with photos:** equipment, corrosion, site imagery. That's the industrial computer vision scoping talk, next.

## Facilitator: save this run as the room's fallback

In [ ]:
from vision_client import promote_to_prebaked

PROMOTE = False  # facilitator only: after a good live run, keep it for when the network or a model fails
if PROMOTE and RUN_MODE == "live":
    promote_to_prebaked(OUT / "captions", PREBAKED / "captions")
    promote_to_prebaked(OUT / "answers", PREBAKED / "answers")